# Modulo 9 — Report

Per ogni strategia genera grafici (equity, drawdown, distribuzione profitti), tabella di metriche (Sharpe, Sortino, Calmar, Win Rate, Profit Factor, Expectancy, Recovery Factor, numero trade) e una spiegazione del **perché è stata selezionata** e dei suoi **limiti**.

In [ ]:
import sys
from pathlib import Path
for c in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (c / 'trading_ai').exists():
        sys.path.insert(0, str(c)); break

from trading_ai.data_engine import DataEngine, generate_ohlcv
from trading_ai.feature_engineering import FeatureEngine
from trading_ai.pattern_discovery.clustering import FeatureClusterer
from trading_ai.strategy_generator import Strategy, RiskParams
from trading_ai.validation import Validator
from trading_ai.reporting import ReportGenerator

In [ ]:
eng = DataEngine()
h1 = eng.to_timeframe(eng.load_dataframe(generate_ohlcv(n=150_000, seed=9)), 'H1')
feats = FeatureEngine().transform(h1, dropna=True)
cols = [c for c in feats.columns if c not in ('open','high','low','close','volume')]
clu = FeatureClusterer(n_clusters=8, feature_columns=cols).fit(feats)
strat = Strategy(name='DEMO_LONG', cluster_id=2, direction=1, clusterer=clu,
                 risk=RiskParams(sl_atr=2, tp_atr=3, be_atr=1, trail_atr=1.5, max_bars=40))
result = strat.run(feats)
print('Trade:', len(result.trades))

In [ ]:
# Validazione opzionale (per la sezione 'perché selezionata').
report_val = None
try:
    report_val = Validator(min_trades=20, n_mc_sims=500, n_splits=4).validate(strat, feats)
except Exception as e:
    print('Validazione saltata:', e)

rep = ReportGenerator()
out = rep.generate(strat, result, validation=report_val)
print('Report:', out['markdown'])
print(out['markdown'].read_text()[:1200])

## ✅ Modulo 9 verificato

Grafici, metriche complete e report con razionale e limiti. Test: `pytest tests/test_reporting.py` (7/7).